# Feature Engineering

**Goal:** Transform raw loan data into model-ready features based on credit risk domain knowledge.

## Feature Groups
1. **Core risk metrics** — FICO, DTI, LTV (used as-is)
2. **Ratio features** — derived financial ratios that capture leverage and affordability
3. **Binary flags** — categorical variables encoded for modeling
4. **Risk composite** — combined high-risk indicator

All features are grounded in standard mortgage underwriting logic.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from src.utils import load_data

df = load_data(data_dir='../data', years=[2022, 2023])
print(f"Loaded: {df.shape}")

Loading origination data...
  Loaded origination 2022: 50,000 loans
  Loaded origination 2023: 50,000 loans

Cleaning origination data...

Loading performance data...
  Loaded performance 2022: 1,788,639 records
  Loaded performance 2023: 1,223,737 records

Creating target variable...
Default rate: 2.43%

Done! Dataset: (100000, 31)
Loaded: (100000, 31)


## 1. Ratio Features
Financial ratios that capture affordability and leverage — standard metrics in credit underwriting.

In [2]:
# Equity at origination — how much skin-in-the-game the borrower has
# Higher equity = lower default risk (borrower has more to lose)
df['equity_pct'] = 100 - df['oltv']

# Rate spread — how much above/below the yearly median this loan's rate is
# Positive spread may indicate higher-risk borrower priced accordingly
df['rate_spread'] = (df['orig_interest_rate'] 
                     - df.groupby('year')['orig_interest_rate'].transform('median'))

# Loan-to-income proxy — larger loan relative to term = higher monthly burden
# (true income not available, so we use loan size / term as proxy)
df['upb_per_month'] = df['orig_upb'] / df['orig_loan_term']

print("Ratio features created:")
print(df[['equity_pct', 'rate_spread', 'upb_per_month']].describe().round(2))

Ratio features created:
       equity_pct  rate_spread  upb_per_month
count   100000.00    100000.00      100000.00
mean        25.27        -0.09         900.98
std         19.00         1.00         511.40
min          3.00        -3.88          55.56
25%         10.00        -0.75         527.78
50%         20.00         0.00         797.22
75%         35.00         0.62        1166.67
max         97.00         2.88        6244.44


## 2. Binary Flags — Loan Purpose & Borrower
Categorical variables encoded as 0/1 based on EDA findings.

In [3]:
# Loan purpose — from EDA: cash-out refi shows higher default risk
df['is_purchase']      = (df['loan_purpose'] == 'P').astype(int)
df['is_cashout_refi']  = (df['loan_purpose'] == 'C').astype(int)

# First time homebuyer
df['first_time_homebuyer_flag'] = (df['first_time_homebuyer'] == 'Y').astype(int)

# Number of borrowers — 2 borrowers = lower default risk (EDA confirmed)
df['is_joint_loan'] = (df['num_borrowers'] == 2).astype(int)

print("Loan purpose distribution:")
print(df[['is_purchase', 'is_cashout_refi']].mean().round(3))

Loan purpose distribution:
is_purchase        0.753
is_cashout_refi    0.184
dtype: float64


## 3. Binary Flags — Channel, Property & Occupancy
EDA revealed significant default rate differences across these categories.

In [4]:
# Origination channel — EDA: Broker (2.63%) > Retail (2.59%) > Correspondent (2.12%)
df['is_broker']        = (df['channel'] == 'B').astype(int)
df['is_correspondent'] = (df['channel'] == 'C').astype(int)

# Property type — EDA: Mobile Home default rate 4.20% vs portfolio avg 2.43%
df['is_mobile_home'] = (df['property_type'] == 'MH').astype(int)
df['is_condo']       = (df['property_type'] == 'CO').astype(int)

# Occupancy — investment properties historically show higher default rates
df['is_investment']  = (df['occupancy_status'] == 'I').astype(int)
df['is_second_home'] = (df['occupancy_status'] == 'S').astype(int)

# Loan term — 30yr vs 15yr
df['is_30yr'] = (df['orig_loan_term'] == 360).astype(int)

print("New categorical flags — default rates:")
flags = ['is_broker', 'is_correspondent', 'is_mobile_home',
         'is_condo', 'is_investment', 'is_second_home']
for f in flags:
    dr = df[df[f]==1]['default'].mean() * 100
    print(f"  {f:22s}: {dr:.2f}%")

New categorical flags — default rates:
  is_broker             : 2.63%
  is_correspondent      : 2.12%
  is_mobile_home        : 4.20%
  is_condo              : 1.91%
  is_investment         : 1.02%
  is_second_home        : 1.11%


## 4. Composite Risk Flag
Combined indicator capturing borrowers with multiple simultaneous risk factors.

In [5]:
# High risk = low FICO + high LTV + high DTI simultaneously
# Each threshold based on standard underwriting guidelines
df['high_risk'] = (
    (df['credit_score'] < 680) &   # below fair credit threshold
    (df['oltv'] > 90) &             # minimal equity buffer
    (df['dti'] > 43)                # above QM safe harbor
).astype(int)

high_risk_dr = df[df['high_risk']==1]['default'].mean() * 100
normal_dr    = df[df['high_risk']==0]['default'].mean() * 100
print(f"High risk loans:   {df['high_risk'].sum():,} ({df['high_risk'].mean()*100:.1f}%)")
print(f"Default rate — high risk: {high_risk_dr:.2f}%")
print(f"Default rate — normal:    {normal_dr:.2f}%")

High risk loans:   387 (0.4%)
Default rate — high risk: 11.63%
Default rate — normal:    2.40%


## 5. Final Feature Selection & Export


In [6]:
feature_cols = [
    # Core risk metrics
    'credit_score', 'dti', 'oltv', 'ocltv', 'orig_interest_rate',
    # Loan characteristics
    'orig_upb', 'orig_loan_term', 'num_borrowers',
    # Ratio features
    'equity_pct', 'rate_spread', 'upb_per_month',
    # Loan purpose
    'is_purchase', 'is_cashout_refi',
    # Borrower
    'first_time_homebuyer_flag', 'is_joint_loan',
    # Channel — EDA insight
    'is_broker', 'is_correspondent',
    # Property — EDA insight
    'is_mobile_home', 'is_condo',
    # Occupancy — EDA insight
    'is_investment', 'is_second_home',
    # Loan term
    'is_30yr',
    # Composite risk flag
    'high_risk',
    # Vintage
    'year',
    # Target
    'default'
]

df_model = df[feature_cols].copy()

# Fill missing with median
num_cols = df_model.select_dtypes(include='number').columns.drop('default')
df_model[num_cols] = df_model[num_cols].fillna(df_model[num_cols].median())

df_model.to_csv('../data/modeling_data.csv', index=False)

print(f"modeling_data.csv saved!")
print(f"Shape: {df_model.shape}")
print(f"Features: {df_model.shape[1]-1}")
print(f"Missing values: {df_model.isnull().sum().sum()}")
print(f"Default rate: {df_model['default'].mean()*100:.2f}%")

modeling_data.csv saved!
Shape: (100000, 25)
Features: 24
Missing values: 0
Default rate: 2.43%
